# 2 — Kaggle submission: your trained Gemma 2 2B + LoRA

**Competition: LLM Classification Finetuning (`llm-classification-finetuning`).**
This is **not** a Gemma 4 Developer Agent submission.

## One-time setup
1. Locally run `1_export_arena_model.ipynb`. Upload its `arena_lora_2b_assets.zip` as a **private Kaggle Dataset**.
2. Import this notebook using **Your Work → Create → Import Notebook**.
3. In the editor, use **Add Input** to attach **both** that private dataset and the **LLM Classification Finetuning competition data**. Accept the competition rules when prompted.
4. Select a **GPU accelerator**, and set **Internet OFF**. A native-BF16 GPU is preferred when available; older GPUs use the FP32 path below.
5. **Save Version → Save & Run All**. Once the saved version succeeds, submit its **`submission.csv`** to **LLM Classification Finetuning** from the output/competition submission panel.

**No training occurs here.** The notebook loads the original base checkpoint and the saved best adapter **including the trained head**.
It preserves the training input format and 1,536-token maximum. No quantization, merging, temperature scaling, swapping, or ensembling is added.
Only the current competition `test.csv` supplies prediction rows: the three public examples are replaced by the full hidden test during scoring.

Importing an `.ipynb` uploads code, **not the model files from your Windows computer**.

## 1. Locate your attached model bundle
No paths normally need editing. If multiple matching bundles are attached, set `ASSET_DIR` to the intended bundle directory.
If Kaggle leaves the uploaded ZIP unexpanded, it is safely extracted into `/kaggle/working`.

In [ ]:
import json
import os
import sys
import zipfile
from pathlib import Path

ASSET_DIR = None   # Normally auto-detected from the attached private dataset.
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

if ASSET_DIR is None:
    manifests = list(INPUT_ROOT.rglob("arena_manifest.json"))
    if not manifests:
        archives = list(INPUT_ROOT.rglob("arena_lora_2b_assets.zip"))
        if len(archives) == 1:
            extract_dir = WORK_DIR / "_arena_assets"
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as z:
                for member in z.infolist():
                    target = (extract_dir / member.filename).resolve()
                    if not target.is_relative_to(extract_dir.resolve()):
                        raise ValueError("Unsafe archive path.")
                z.extractall(extract_dir)
            manifests = list(extract_dir.rglob("arena_manifest.json"))
    if len(manifests) != 1:
        raise FileNotFoundError(
            f"Found {len(manifests)} model bundles. Attach the private dataset created by the exporter; "
            "if several are attached, explicitly set ASSET_DIR."
        )
    ASSET_DIR = manifests[0].parent
else:
    ASSET_DIR = Path(ASSET_DIR)

manifest = json.loads((ASSET_DIR / "arena_manifest.json").read_text())
assert manifest["bundle_format"] == "arena-gemma2-lora-v1"
assert manifest["competition"] == "llm-classification-finetuning"
assert manifest["model_name"] == "google/gemma-2-2b-it"
BASE_DIR = ASSET_DIR / manifest["base_subdir"]
ADAPTER_DIR = ASSET_DIR / manifest["adapter_subdir"]
print("Model bundle:", ASSET_DIR)
print("Recorded cached base snapshot:", manifest["base_snapshot"])
print("Original reloaded validation log loss:", manifest["reloaded_validation_log_loss"])

## 2. Install the bundled model libraries offline
This cell runs before importing Transformers/PEFT. Wheels are installed into a separate working directory;
Kaggle's PyTorch, CUDA, NumPy, and pandas are preserved. No internet downloads occur in this notebook.
After changing the attached assets or installing other model libraries interactively, restart the session and run from the top.

In [ ]:
import importlib
import subprocess
from importlib import metadata

python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
if python_version not in manifest["runtimes"]:
    raise RuntimeError(
        f"Kaggle uses Python {python_version}, but the bundle has {list(manifest['runtimes'])}. "
        "Add this Python version to KAGGLE_PYTHON_VERSIONS in the local exporter and re-upload the assets."
    )
runtime = manifest["runtimes"][python_version]
wheel_dir = ASSET_DIR / runtime["wheel_subdir"]
runtime_dir = WORK_DIR / "_arena_runtime"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--no-index", "--find-links", str(wheel_dir), "--no-deps",
    "--target", str(runtime_dir), "--upgrade", "-r", str(wheel_dir / "requirements.txt"),
])
sys.path.insert(0, str(runtime_dir))
importlib.invalidate_caches()

from packaging.requirements import Requirement
for requirement in runtime["preserved_requirements"]:
    req = Requirement(requirement)
    installed = metadata.version(req.name)
    if not req.specifier.contains(installed, prereleases=True):
        raise RuntimeError(f"Kaggle has {req.name}=={installed}, but the bundled stack requires {requirement}.")
print("Offline model libraries installed; Kaggle's CUDA/PyTorch was not replaced.")

## 3. Load the competition data and the saved tokenizer
`test.csv` is always read from the competition input, never from the private model dataset.
All rows are processed; no three-row limit is hardcoded. IDs are kept as strings to preserve their exact values.

In [ ]:
import gc
import hashlib
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from safetensors import safe_open
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

COMPETITION_DIR = INPUT_ROOT / "llm-classification-finetuning"
if not (COMPETITION_DIR / "test.csv").is_file():
    candidates = [p for p in INPUT_ROOT.rglob("test.csv")
                  if "llm-classification-finetuning" in p.parts]
    if len(candidates) != 1:
        raise FileNotFoundError("Use Add Input to attach the LLM Classification Finetuning competition data.")
    COMPETITION_DIR = candidates[0].parent

test_df = pd.read_csv(COMPETITION_DIR / "test.csv", dtype={"id": "string"}, keep_default_na=False)
required_columns = {"id", "prompt", "response_a", "response_b"}
assert required_columns.issubset(test_df.columns)
assert len(test_df) > 0 and test_df["id"].is_unique and test_df["id"].str.len().gt(0).all()
LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
assert manifest["label_columns"] == LABEL_COLUMNS
if (COMPETITION_DIR / "sample_submission.csv").is_file():
    header = pd.read_csv(COMPETITION_DIR / "sample_submission.csv", nrows=0).columns.tolist()
    assert header == ["id", *LABEL_COLUMNS], header

MAX_LENGTH = manifest["max_length"]
PROMPT_BUDGET = manifest["prompt_budget"]
assert (MAX_LENGTH, PROMPT_BUDGET) == (1536, 256)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), local_files_only=True)
tokenizer.padding_side = "right"
assert tokenizer.bos_token_id is not None and tokenizer.eos_token_id is not None
assert tokenizer.pad_token_id is not None
print("Competition input:", COMPETITION_DIR)
print(f"Rows to predict: {len(test_df):,}")
print({p: metadata.version(p) for p in ["torch", "transformers", "peft", "tokenizers"]})

## 4. The exact training preprocessing
These functions are copied from the executed training notebook. In particular, this does not replace the template
with `apply_chat_template`, flatten turn markers, change answer order, or change truncation.

In [ ]:
def parse_turns(value):
    """Decode one competition field. Never evaluate Python code from the data."""
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list):
        raise ValueError("Expected a JSON list of turns.")
    if not all(turn is None or isinstance(turn, str) for turn in turns):
        raise ValueError("Turns must be strings or null.")
    return ["" if turn is None else turn for turn in turns] or [""]


def format_turns(turns):
    return "\n\n".join(
        f"[Turn {i + 1}]\n{turn if turn else '[No text]'}"
        for i, turn in enumerate(turns)
    )


def prepare_row(row):
    try:
        p, a, b = [parse_turns(row[col]) for col in ["prompt", "response_a", "response_b"]]
    except (ValueError, TypeError) as exc:
        raise ValueError(f"Invalid conversation in row id={row['id']}: {exc}") from exc
    # Group identical normalized full prompt conversations, not individual response pairs.
    normalized = json.dumps([" ".join(turn.split()) for turn in p], ensure_ascii=False)
    group = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    n_turns = max(len(p), len(a), len(b))
    mismatched = len({len(p), len(a), len(b)}) > 1
    p, a, b = [turns + [""] * (n_turns - len(turns)) for turns in [p, a, b]]
    return {
        "prompt_text": format_turns(p),
        "response_a_text": format_turns(a),
        "response_b_text": format_turns(b),
        "prompt_group": group,
        "num_turns": n_turns,
        "mismatched_turns": mismatched,
    }


def encode_text(text):
    return tokenizer.encode(text, add_special_tokens=False)


prefix_ids = encode_text("Choose which response a human would prefer: A, B, or tie.\n\n[Prompt]\n")
a_header_ids = encode_text("\n\n[Response A]\n")
b_header_ids = encode_text("\n\n[Response B]\n")
suffix_ids = encode_text("\n\n[Decision: A, B, or tie]")
omission_ids = encode_text("\n[... omitted ...]\n")
overhead = len(prefix_ids) + len(a_header_ids) + len(b_header_ids) + len(suffix_ids) + 2
assert MAX_LENGTH % 8 == 0 and MAX_LENGTH <= tokenizer.model_max_length
assert MAX_LENGTH - overhead - PROMPT_BUDGET > 2 * len(omission_ids)


def trim_tokens(ids, budget):
    """A token budget for ONE example, with head-and-tail preservation."""
    if len(ids) <= budget:
        return ids
    if budget <= len(omission_ids):
        return ids[:budget]
    remaining = budget - len(omission_ids)
    head = (remaining + 1) // 2
    tail = remaining // 2
    return ids[:head] + omission_ids + (ids[-tail:] if tail else [])


def assemble_input(p, a, b):
    p_budget = min(len(p), PROMPT_BUDGET)
    answer_space = MAX_LENGTH - overhead - p_budget
    a_budget = min(len(a), answer_space // 2)
    b_budget = min(len(b), answer_space // 2)
    # Redistribute unused answer space without ever dropping the other answer's allocation.
    spare = answer_space - a_budget - b_budget
    extra_a = min(spare, len(a) - a_budget)
    a_budget += extra_a
    b_budget += min(spare - extra_a, len(b) - b_budget)
    input_ids = (
        [tokenizer.bos_token_id] + prefix_ids + trim_tokens(p, p_budget)
        + a_header_ids + trim_tokens(a, a_budget)
        + b_header_ids + trim_tokens(b, b_budget)
        + suffix_ids + [tokenizer.eos_token_id]
    )
    assert len(input_ids) <= MAX_LENGTH
    return input_ids, [len(p) > p_budget, len(a) > a_budget, len(b) > b_budget]


def tokenize(batch):
    # Each of these is a batch of complete example strings; never slice the row list as a token limit.
    p_batch = tokenizer(batch["prompt_text"], add_special_tokens=False, truncation=False)["input_ids"]
    a_batch = tokenizer(batch["response_a_text"], add_special_tokens=False, truncation=False)["input_ids"]
    b_batch = tokenizer(batch["response_b_text"], add_special_tokens=False, truncation=False)["input_ids"]
    result = {key: [] for key in ["input_ids", "attention_mask", "length", "original_length",
                                  "truncated_prompt", "truncated_a", "truncated_b"]}
    for p, a, b in zip(p_batch, a_batch, b_batch):
        ids, flags = assemble_input(p, a, b)
        result["input_ids"].append(ids)
        result["attention_mask"].append([1] * len(ids))
        result["length"].append(len(ids))
        result["original_length"].append(overhead + len(p) + len(a) + len(b))
        for name, flag in zip(["truncated_prompt", "truncated_a", "truncated_b"], flags):
            result[name].append(flag)
    if "labels" in batch:
        result["labels"] = batch["labels"]
    return result

In [ ]:
def encode_records(records, batch_size=128):
    """Encode raw CSV records with the training notebook's exact functions."""
    encoded = []
    for start in range(0, len(records), batch_size):
        prepared = [prepare_row(row) for row in records[start:start + batch_size]]
        batch = {key: [row[key] for row in prepared] for key in
                 ["prompt_text", "response_a_text", "response_b_text"]}
        result = tokenize(batch)
        encoded.extend({"input_ids": ids, "attention_mask": mask}
                       for ids, mask in zip(result["input_ids"], result["attention_mask"]))
    return encoded

checks = json.loads((ASSET_DIR / "tokenization_checks.json").read_text(encoding="utf-8"))
for row, encoded in zip(checks, encode_records(checks)):
    digest = hashlib.sha256(json.dumps(encoded["input_ids"], separators=(",", ":")).encode()).hexdigest()
    assert len(encoded["input_ids"]) == row["expected_length"] and digest == row["expected_sha256"], (
        f"Tokenization differs from the exported training tokenizer: {row['id']}"
    )
print("Training-tokenizer fingerprint checks passed.")

start_tokenization = time.perf_counter()
records = test_df[["id", "prompt", "response_a", "response_b"]].to_dict("records")
features = encode_records(records)
lengths = np.asarray([len(row["input_ids"]) for row in features])
assert len(features) == len(test_df) and lengths.max() <= MAX_LENGTH
print(f"Tokenized in {time.perf_counter() - start_tokenization:.1f}s; "
      f"mean length {lengths.mean():.1f}, maximum {lengths.max()}.")
del records

## 5. Load the frozen base and the trained adapter/head
Use **native BF16** on GPUs that support it. Otherwise use **FP32**, not an unvalidated FP16 conversion.
The FP32 path needs more memory and may be slower; its probabilities can differ slightly from the local BF16 evaluation.
This notebook uses one GPU even when two are allocated. No multi-GPU sharding or quantization is introduced.

The base checkpoint does not contain a trained `score` head. A missing-head warning **at base loading** is expected;
the following PEFT load replaces it. An exact saved-versus-loaded head check prevents submitting a random head.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Session options before running this notebook.")
DEVICE = torch.device("cuda:0")
try:
    native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    native_bf16 = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if native_bf16 else torch.float32
BATCH_SIZE = 4 if native_bf16 else 2  # Automatically reduced if an actual batch runs out of memory.
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", MODEL_DTYPE, "| batch:", BATCH_SIZE)

for name, size in manifest["base_files"].items():
    path = BASE_DIR / name
    assert path.is_file() and path.stat().st_size == size, f"Missing or incomplete base checkpoint: {path}"
adapter_path = ADAPTER_DIR / "adapter_model.safetensors"
assert hashlib.sha256(adapter_path.read_bytes()).hexdigest() == manifest["adapter_sha256"], "Adapter file changed."
adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
assert adapter_config["task_type"] == "SEQ_CLS" and "score" in (adapter_config.get("modules_to_save") or [])
with safe_open(str(adapter_path), framework="pt", device="cpu") as f:
    head_keys = [k for k in f.keys() if "score" in k and k.endswith("weight")]
    assert len(head_keys) == 1
    saved_head = f.get_tensor(head_keys[0]).float()
assert saved_head.shape == (3, 2304) and torch.isfinite(saved_head).all()

label_names = ["win_a", "win_b", "tie"]
base_model = AutoModelForSequenceClassification.from_pretrained(
    str(BASE_DIR), local_files_only=True,
    num_labels=3, problem_type="single_label_classification",
    id2label=dict(enumerate(label_names)), label2id={v: i for i, v in enumerate(label_names)},
    pad_token_id=tokenizer.pad_token_id,
    dtype=MODEL_DTYPE, attn_implementation="eager",
)
base_model.config.use_cache = False
base_model.score.float()
model = PeftModel.from_pretrained(
    base_model, str(ADAPTER_DIR), is_trainable=False,
    autocast_adapter_dtype=True, local_files_only=True,
)
loaded_head = model.get_base_model().score.modules_to_save["default"].weight.detach().cpu().float()
torch.testing.assert_close(loaded_head, saved_head, rtol=0, atol=0)
assert any("lora_B" in name for name, _ in model.named_parameters())
model.requires_grad_(False)
model.to(DEVICE).eval()
model.config.use_cache = False
del saved_head, loaded_head, base_model
gc.collect()
torch.cuda.reset_peak_memory_stats()
print("Saved classification head verified. Base model + trained LoRA loaded for inference.")

## 6. Predict every test row
Batches are sorted by length to reduce padding; predictions are restored to their original row positions.
Out-of-memory handling retries with a smaller batch but **never skips rows or substitutes default predictions**.
Only three-class logits are requested. We do not generate text or allocate a generation KV cache.

In [ ]:
@torch.inference_mode()
def predict_batch(indices):
    batch = tokenizer.pad([features[int(i)] for i in indices], padding=True,
                          pad_to_multiple_of=8, return_tensors="pt")
    batch = {name: value.to(DEVICE) for name, value in batch.items()}
    context = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if native_bf16 else nullcontext()
    with context:
        logits = model(**batch, use_cache=False, return_dict=True).logits
    return logits.float().cpu().numpy().astype(np.float64)


order = np.argsort(-lengths, kind="stable")  # Longest batches first; exposes memory issues early.
all_logits = np.full((len(features), 3), np.nan, dtype=np.float64)
batch_size = BATCH_SIZE
position = 0
prediction_start = time.perf_counter()
with tqdm(total=len(features), desc="Predict test rows") as progress:
    while position < len(order):
        indices = order[position:position + batch_size]
        try:
            batch_logits = predict_batch(indices)
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise RuntimeError("One example does not fit. Use a GPU with more free memory; no rows were skipped.")
            batch_size = max(1, batch_size // 2)
            gc.collect()
            torch.cuda.empty_cache()
            print("Reduced inference batch size to", batch_size)
            continue
        if batch_logits.shape != (len(indices), 3) or not np.isfinite(batch_logits).all():
            raise ValueError("Invalid model logits. Do not submit until the model/precision issue is resolved.")
        all_logits[indices] = batch_logits
        position += len(indices)
        progress.update(len(indices))

assert np.isfinite(all_logits).all(), "Some test examples were not predicted."
# Stable float64 softmax, matching the training notebook's probability calculation.
shifted = all_logits - all_logits.max(axis=1, keepdims=True)
probabilities = np.exp(shifted)
probabilities /= probabilities.sum(axis=1, keepdims=True)
assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1.0)
prediction_seconds = time.perf_counter() - prediction_start
print(f"Prediction time: {prediction_seconds / 60:.1f} minutes")

## 7. Write and validate the required submission
The public-example comparison is diagnostic only: saved reference predictions are **never** used to fill the submission.
Different GPU arithmetic and padding batches need not be bit-identical to local BF16 evaluation.
When scoring supplies different IDs, the reference comparison is skipped automatically.

In [ ]:
submission = pd.DataFrame(probabilities, columns=LABEL_COLUMNS)
submission.insert(0, "id", test_df["id"].to_numpy())
assert submission.columns.tolist() == ["id", *LABEL_COLUMNS]
assert len(submission) == len(test_df) and submission["id"].is_unique
assert np.array_equal(submission["id"].astype(str), test_df["id"].astype(str))
assert submission[LABEL_COLUMNS].ge(0).all().all() and submission[LABEL_COLUMNS].le(1).all().all()

reference_difference = None
reference_path = ASSET_DIR / "public_test_reference.csv"
if reference_path.is_file():
    reference = pd.read_csv(reference_path, dtype={"id": "string"})
    if len(reference) == len(submission) and set(reference["id"]) == set(submission["id"]):
        expected = reference.set_index("id").loc[submission["id"], LABEL_COLUMNS].to_numpy()
        reference_difference = float(np.max(np.abs(expected - probabilities)))
        print("Public-example maximum probability difference from local run:", reference_difference)
        if reference_difference > 0.05:
            print("WARNING: investigate the model snapshot, precision, and versions before submission.")

submission_path = WORK_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
# Check the actual on-disk artifact, not just the DataFrame.
written = pd.read_csv(submission_path, dtype={"id": "string"})
assert written.columns.tolist() == ["id", *LABEL_COLUMNS] and len(written) == len(test_df)
assert np.array_equal(written["id"].astype(str), test_df["id"].astype(str))
assert np.isfinite(written[LABEL_COLUMNS].to_numpy()).all()
assert np.allclose(written[LABEL_COLUMNS].sum(axis=1), 1.0, atol=1e-7)

inference_summary = {
    "competition": "llm-classification-finetuning",
    "test_rows": len(test_df), "model_name": manifest["model_name"],
    "base_snapshot": manifest["base_snapshot"], "adapter_sha256": manifest["adapter_sha256"],
    "gpu": torch.cuda.get_device_name(0), "dtype": str(MODEL_DTYPE),
    "batch_size_after_memory_check": batch_size, "max_length": MAX_LENGTH,
    "prediction_seconds": prediction_seconds,
    "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
    "public_reference_max_difference": reference_difference,
    "versions": {p: metadata.version(p) for p in ["torch", "transformers", "peft", "tokenizers"]},
}
(WORK_DIR / "inference_summary.json").write_text(json.dumps(inference_summary, indent=2))
print("Submission ready:", submission_path)
print(submission.head().to_string(index=False))
print("Save Version → Save & Run All, then submit the saved version's submission.csv to the competition.")

## Submission requirements and scope
Kaggle currently requires an offline notebook, a CPU/GPU runtime of at most **9 hours**, and an output named **`submission.csv`**.
The competition data page says that its demonstration test rows are replaced with the complete hidden test set at scoring time.
A CSV produced from the three local examples is therefore not a standalone hidden-test submission.

This notebook preserves your trained model and preprocessing; it does not promise a particular leaderboard score or runtime.
The recorded 0.96477 is your local holdout log loss, not a Kaggle score. Runtime must be checked on the selected Kaggle accelerator.

**Official references:**
[Competition requirements](https://www.kaggle.com/competitions/llm-classification-finetuning/overview)
· [Hidden-test replacement and file schema](https://www.kaggle.com/competitions/llm-classification-finetuning/data)
· [Kaggle inputs, settings, and saved versions](https://www.kaggle.com/docs/notebooks)
· [PEFT save/load](https://huggingface.co/docs/peft/package_reference/peft_model)
· [PyTorch native BF16 support](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html)